In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/iamaniketdubey/sandbox-data/synthetic_data.csv


In [39]:
df = pd.read_csv("/kaggle/input/datasets/iamaniketdubey/sandbox-data/synthetic_data.csv")

In [40]:
df.sample()

,num_feature_1,num_feature_2,num_feature_3,num_feature_4,num_feature_5,num_feature_6,num_feature_7,num_feature_8,num_feature_9,num_feature_10,...,num_feature_15,num_feature_16,num_feature_17,num_feature_18,cat_region,cat_segment,cat_flag,noise_feature_1,noise_feature_2,target
6548,-1.537441,0.825265,-0.576336,-1.321305,0.591815,-10.003483,1.047311,0.459339,-10.550014,5.448648,...,0.386521,0.901699,2.298511,5.912012,North,A,No,65.810535,61.087464,0


In [41]:
print(df.dtypes)

print(df.isnull().sum())




num_feature_1      float64
num_feature_2      float64
num_feature_3      float64
num_feature_4      float64
num_feature_5      float64
num_feature_6      float64
num_feature_7      float64
num_feature_8      float64
num_feature_9      float64
num_feature_10     float64
num_feature_11     float64
num_feature_12     float64
num_feature_13     float64
num_feature_14     float64
num_feature_15     float64
num_feature_16     float64
num_feature_17     float64
num_feature_18     float64
cat_region          object
cat_segment         object
cat_flag            object
noise_feature_1    float64
noise_feature_2    float64
target               int64
dtype: object
num_feature_1        0
num_feature_2        0
num_feature_3      499
num_feature_4        0
num_feature_5        0
num_feature_6        0
num_feature_7      790
num_feature_8        0
num_feature_9        0
num_feature_10       0
num_feature_11       0
num_feature_12       0
num_feature_13       0
num_feature_14       0
num_feature_15  

In [36]:
df_new = df_new[(df.isnull().sum()/len(df))*100>0]
df_new

Series([], dtype: float64)

In [42]:
numeric_missing_cols = ["num_feature_3", "num_feature_7", "noise_feature_1"]

for col in numeric_missing_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Filled {col} with meidan = {median_val:.3f}")

Filled num_feature_3 with meidan = -0.808
Filled num_feature_7 with meidan = 0.861
Filled noise_feature_1 with meidan = 49.995


In [43]:
df['cat_region'] = df['cat_region'].fillna("Missing")


In [44]:
df.isnull().sum()

num_feature_1      0
num_feature_2      0
num_feature_3      0
num_feature_4      0
num_feature_5      0
num_feature_6      0
num_feature_7      0
num_feature_8      0
num_feature_9      0
num_feature_10     0
num_feature_11     0
num_feature_12     0
num_feature_13     0
num_feature_14     0
num_feature_15     0
num_feature_16     0
num_feature_17     0
num_feature_18     0
cat_region         0
cat_segment        0
cat_flag           0
noise_feature_1    0
noise_feature_2    0
target             0
dtype: int64

In [45]:
df.to_csv("synthetic_data_imputed.csv", index=False)

In [46]:
df_encoded = pd.get_dummies(df, columns=["cat_region", "cat_segment", "cat_flag"], drop_first=True)

In [47]:
print("\nColumns after encoding:", df_encoded.shape[1])
print("\nNew columns created:")
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print(new_cols)

df_encoded.to_csv("synthetic_data_encoded.csv", index=False)


Columns after encoding: 28

New columns created:
['cat_region_Missing', 'cat_region_North', 'cat_region_South', 'cat_region_West', 'cat_segment_B', 'cat_segment_C', 'cat_flag_Yes']


In [49]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [50]:
X = df.drop(columns = ['target'])
y = df['target']

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X,y ,test_size = 0.2, random_state = 42, stratify = y)

In [52]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target ratio:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest target ratio:\n", y_test.value_counts(normalize=True).round(3))


Train shape: (8000, 23)
Test shape: (2000, 23)

Train target ratio:
 target
0    0.647
1    0.353
Name: proportion, dtype: float64

Test target ratio:
 target
0    0.646
1    0.354
Name: proportion, dtype: float64


In [55]:
numeric_cols = [c for c in X.columns if c.startswith("num_feature") or c.startswith("noise_feature")]
print("\nNumeric columns to scale:", len(numeric_cols))


Numeric columns to scale: 20


In [64]:
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols]) 

print("\nTrain num_feature_1 mean (should be ~0):", X_train_scaled["num_feature_1"].mean())
print("Train num_feature_1 std (should be ~1):", X_train_scaled["num_feature_1"].std())
print("Test num_feature_1 mean (won't be exactly 0, that's expected):", X_test_scaled["num_feature_1"].mean())


Train num_feature_1 mean (should be ~0): 5.3290705182007515e-18
Train num_feature_1 std (should be ~1): 1.0000625058599857
Test num_feature_1 mean (won't be exactly 0, that's expected): 0.009435389292023632


In [65]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("synthetic_data_encoded.csv")

# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Step 1: Split FIRST, before any scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # preserves class ratio in both splits
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target ratio:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest target ratio:\n", y_test.value_counts(normalize=True).round(3))

# Step 2: Identify which columns actually need scaling
# (one-hot encoded columns are already 0/1, no need to scale them)
numeric_cols = [c for c in X.columns if c.startswith("num_feature") or c.startswith("noise_feature")]
print("\nNumeric columns to scale:", len(numeric_cols))

# Step 3: Fit scaler on TRAIN ONLY
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])  # transform only, no fit!

# Quick sanity check
print("\nTrain num_feature_1 mean (should be ~0):", X_train_scaled["num_feature_1"].mean())
print("Train num_feature_1 std (should be ~1):", X_train_scaled["num_feature_1"].std())
print("Test num_feature_1 mean (won't be exactly 0, that's expected):", X_test_scaled["num_feature_1"].mean())

Train shape: (8000, 27)
Test shape: (2000, 27)

Train target ratio:
 target
0    0.647
1    0.353
Name: proportion, dtype: float64

Test target ratio:
 target
0    0.646
1    0.354
Name: proportion, dtype: float64

Numeric columns to scale: 20

Train num_feature_1 mean (should be ~0): 5.3290705182007515e-18
Train num_feature_1 std (should be ~1): 1.0000625058599857
Test num_feature_1 mean (won't be exactly 0, that's expected): 0.009435389292023632


In [66]:
 X_train_scaled["num_feature_1"]

1855    0.581791
2756   -0.673812
7572    1.010429
1611   -1.196428
1260    1.065018
          ...   
4798   -0.089362
9942    0.212860
9103    0.278774
1391    0.501443
510    -0.191453
Name: num_feature_1, Length: 8000, dtype: float64

In [67]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


In [69]:
knn = KNeighborsClassifier(n_neighbors = 5)

In [70]:
knn.fit(X_train_scaled, y_train)

KNeighborsClassifier()

In [71]:
y_pred = knn.predict(X_test_scaled)

In [72]:
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nFull Classification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.9255
Precision: 0.9253
Recall: 0.8586
F1 Score: 0.8907

Confusion Matrix:
[[1244   49]
 [ 100  607]]

Full Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94      1293
           1       0.93      0.86      0.89       707

    accuracy                           0.93      2000
   macro avg       0.93      0.91      0.92      2000
weighted avg       0.93      0.93      0.92      2000



In [73]:
from sklearn.model_selection import cross_val_score

In [75]:
import numpy as np
k_values =range(1,31)

mean_f1_score = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors = k)
    cv_score =  cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='f1')
    mean_f1_score.append(cv_score.mean())

In [79]:

# Find the best k
best_k = k_values[np.argmax(mean_f1_score)]
best_score = max(mean_f1_score)

print("Best k:", best_k)
print("Best mean F1 (cross-validated):", round(best_score, 4))

print("\nAll scores:")
for k, score in zip(k_values, mean_f1_score):
    print(f"k={k:2d} -> F1={score:.4f}")

Best k: 11
Best mean F1 (cross-validated): 0.8881

All scores:
k= 1 -> F1=0.8364
k= 2 -> F1=0.8163
k= 3 -> F1=0.8773
k= 4 -> F1=0.8548
k= 5 -> F1=0.8796
k= 6 -> F1=0.8657
k= 7 -> F1=0.8860
k= 8 -> F1=0.8720
k= 9 -> F1=0.8863
k=10 -> F1=0.8756
k=11 -> F1=0.8881
k=12 -> F1=0.8761
k=13 -> F1=0.8862
k=14 -> F1=0.8744
k=15 -> F1=0.8857
k=16 -> F1=0.8754
k=17 -> F1=0.8832
k=18 -> F1=0.8752
k=19 -> F1=0.8836
k=20 -> F1=0.8754
k=21 -> F1=0.8826
k=22 -> F1=0.8743
k=23 -> F1=0.8837
k=24 -> F1=0.8741
k=25 -> F1=0.8824
k=26 -> F1=0.8727
k=27 -> F1=0.8813
k=28 -> F1=0.8703
k=29 -> F1=0.8781
k=30 -> F1=0.8710


In [80]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Train final model with tuned k
best_knn = KNeighborsClassifier(n_neighbors=11)
best_knn.fit(X_train_scaled, y_train)

y_pred_tuned = best_knn.predict(X_test_scaled)

print("=== Tuned KNN (k=11) — Test Set Performance ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_tuned), 4))
print("Precision:", round(precision_score(y_test, y_pred_tuned), 4))
print("Recall:", round(recall_score(y_test, y_pred_tuned), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_tuned), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\n=== Comparison: Baseline (k=5) vs Tuned (k=11) ===")
print(f"{'Metric':<12}{'k=5':<10}{'k=11':<10}")
print(f"{'Accuracy':<12}{0.9255:<10}{round(accuracy_score(y_test, y_pred_tuned),4):<10}")
print(f"{'Precision':<12}{0.9253:<10}{round(precision_score(y_test, y_pred_tuned),4):<10}")
print(f"{'Recall':<12}{0.8586:<10}{round(recall_score(y_test, y_pred_tuned),4):<10}")
print(f"{'F1 Score':<12}{0.8907:<10}{round(f1_score(y_test, y_pred_tuned),4):<10}")

=== Tuned KNN (k=11) — Test Set Performance ===
Accuracy: 0.9265
Precision: 0.9416
Recall: 0.8444
F1 Score: 0.8904

Confusion Matrix:
[[1256   37]
 [ 110  597]]

=== Comparison: Baseline (k=5) vs Tuned (k=11) ===
Metric      k=5       k=11      
Accuracy    0.9255    0.9265    
Precision   0.9253    0.9416    
Recall      0.8586    0.8444    
F1 Score    0.8907    0.8904    


In [81]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

k_values = range(1, 31)
mean_recall_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='recall')
    mean_recall_scores.append(cv_scores.mean())

best_k_recall = k_values[np.argmax(mean_recall_scores)]
best_recall_score = max(mean_recall_scores)

print("Best k (optimized for recall):", best_k_recall)
print("Best mean recall (cross-validated):", round(best_recall_score, 4))

print("\nAll scores:")
for k, score in zip(k_values, mean_recall_scores):
    print(f"k={k:2d} -> Recall={score:.4f}")

Best k (optimized for recall): 3
Best mean recall (cross-validated): 0.8451

All scores:
k= 1 -> Recall=0.8221
k= 2 -> Recall=0.7131
k= 3 -> Recall=0.8451
k= 4 -> Recall=0.7733
k= 5 -> Recall=0.8369
k= 6 -> Recall=0.7892
k= 7 -> Recall=0.8355
k= 8 -> Recall=0.7980
k= 9 -> Recall=0.8330
k=10 -> Recall=0.8012
k=11 -> Recall=0.8313
k=12 -> Recall=0.8008
k=13 -> Recall=0.8253
k=14 -> Recall=0.7987
k=15 -> Recall=0.8228
k=16 -> Recall=0.7998
k=17 -> Recall=0.8178
k=18 -> Recall=0.7994
k=19 -> Recall=0.8171
k=20 -> Recall=0.7987
k=21 -> Recall=0.8143
k=22 -> Recall=0.7955
k=23 -> Recall=0.8139
k=24 -> Recall=0.7955
k=25 -> Recall=0.8129
k=26 -> Recall=0.7924
k=27 -> Recall=0.8093
k=28 -> Recall=0.7888
k=29 -> Recall=0.8047
k=30 -> Recall=0.7899


In [82]:
best_knn_recall = KNeighborsClassifier(n_neighbors=3)
best_knn_recall.fit(X_train_scaled, y_train)
y_pred_recall = best_knn_recall.predict(X_test_scaled)

print("=== k=3 (recall-optimized) — Test Set Performance ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_recall), 4))
print("Precision:", round(precision_score(y_test, y_pred_recall), 4))
print("Recall:", round(recall_score(y_test, y_pred_recall), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_recall), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_recall))

print("\n=== Full comparison: k=5 vs k=11 (F1-opt) vs k=3 (recall-opt) ===")
print(f"{'Metric':<12}{'k=5':<10}{'k=11':<10}{'k=3':<10}")
print(f"{'Accuracy':<12}{0.9255:<10}{0.9265:<10}{round(accuracy_score(y_test, y_pred_recall),4):<10}")
print(f"{'Precision':<12}{0.9253:<10}{0.9416:<10}{round(precision_score(y_test, y_pred_recall),4):<10}")
print(f"{'Recall':<12}{0.8586:<10}{0.8444:<10}{round(recall_score(y_test, y_pred_recall),4):<10}")
print(f"{'F1 Score':<12}{0.8907:<10}{0.8904:<10}{round(f1_score(y_test, y_pred_recall),4):<10}")

=== k=3 (recall-optimized) — Test Set Performance ===
Accuracy: 0.9115
Precision: 0.8943
Recall: 0.8501
F1 Score: 0.8716

Confusion Matrix:
[[1222   71]
 [ 106  601]]

=== Full comparison: k=5 vs k=11 (F1-opt) vs k=3 (recall-opt) ===
Metric      k=5       k=11      k=3       
Accuracy    0.9255    0.9265    0.9115    
Precision   0.9253    0.9416    0.8943    
Recall      0.8586    0.8444    0.8501    
F1 Score    0.8907    0.8904    0.8716    


In [83]:
import pandas as pd

# Recompute correlation with target using TRAINING data only
train_check = X_train.copy()
train_check["target"] = y_train.values

numeric_cols = [c for c in X_train.columns if c.startswith("num_feature") or c.startswith("noise_feature")]
train_corrs = train_check[numeric_cols + ["target"]].corr()["target"].drop("target")

print("Correlation with target (training data only), sorted by strength:")
print(train_corrs.sort_values(key=abs, ascending=False))

Correlation with target (training data only), sorted by strength:
num_feature_18    -0.545348
num_feature_13    -0.371321
num_feature_7     -0.319071
num_feature_17    -0.281159
num_feature_11     0.252948
num_feature_10     0.249914
num_feature_3      0.248668
num_feature_14     0.237401
num_feature_4      0.119728
num_feature_6      0.100860
num_feature_9     -0.028211
num_feature_16    -0.026038
num_feature_1      0.019810
num_feature_8      0.012384
num_feature_15    -0.012064
num_feature_12     0.009280
num_feature_5     -0.004051
noise_feature_1    0.003574
noise_feature_2   -0.002155
num_feature_2     -0.000135
Name: target, dtype: float64


In [84]:
# Drop near-zero-correlation columns (|r| < 0.05, training-data-based decision)
cols_to_drop = ["num_feature_9", "num_feature_16", "num_feature_1", "num_feature_8",
                "num_feature_15", "num_feature_12", "num_feature_5",
                "noise_feature_1", "noise_feature_2", "num_feature_2"]

X_train_selected = X_train_scaled.drop(columns=cols_to_drop)
X_test_selected = X_test_scaled.drop(columns=cols_to_drop)

print("Original feature count:", X_train_scaled.shape[1])
print("After feature selection:", X_train_selected.shape[1])
print("\nRemaining columns:")
print(list(X_train_selected.columns))


Original feature count: 27
After feature selection: 17

Remaining columns:
['num_feature_3', 'num_feature_4', 'num_feature_6', 'num_feature_7', 'num_feature_10', 'num_feature_11', 'num_feature_13', 'num_feature_14', 'num_feature_17', 'num_feature_18', 'cat_region_Missing', 'cat_region_North', 'cat_region_South', 'cat_region_West', 'cat_segment_B', 'cat_segment_C', 'cat_flag_Yes']


In [85]:
knn_selected = KNeighborsClassifier(n_neighbors=5)
knn_selected.fit(X_train_selected, y_train)

y_pred_selected = knn_selected.predict(X_test_selected)

print("=== KNN (k=5) — Feature-Selected (17 features) — Test Set Performance ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_selected), 4))
print("Precision:", round(precision_score(y_test, y_pred_selected), 4))
print("Recall:", round(recall_score(y_test, y_pred_selected), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_selected), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_selected))

print("\n=== Final comparison: k=5 (27 features) vs k=5 (17 features, selected) ===")
print(f"{'Metric':<12}{'27 features':<14}{'17 features':<14}")
print(f"{'Accuracy':<12}{0.9255:<14}{round(accuracy_score(y_test, y_pred_selected),4):<14}")
print(f"{'Precision':<12}{0.9253:<14}{round(precision_score(y_test, y_pred_selected),4):<14}")
print(f"{'Recall':<12}{0.8586:<14}{round(recall_score(y_test, y_pred_selected),4):<14}")
print(f"{'F1 Score':<12}{0.8907:<14}{round(f1_score(y_test, y_pred_selected),4):<14}")

=== KNN (k=5) — Feature-Selected (17 features) — Test Set Performance ===
Accuracy: 0.9175
Precision: 0.9169
Recall: 0.843
F1 Score: 0.8784

Confusion Matrix:
[[1239   54]
 [ 111  596]]

=== Final comparison: k=5 (27 features) vs k=5 (17 features, selected) ===
Metric      27 features   17 features   
Accuracy    0.9255        0.9175        
Precision   0.9253        0.9169        
Recall      0.8586        0.843         
F1 Score    0.8907        0.8784        


In [86]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train_scaled.columns).sort_values(ascending=False)

print("Mutual Information scores (training data only), sorted:")
print(mi_series)

Mutual Information scores (training data only), sorted:
num_feature_18        0.177807
num_feature_13        0.082607
num_feature_7         0.082070
num_feature_3         0.051918
num_feature_17        0.046566
num_feature_11        0.044352
num_feature_10        0.036267
num_feature_14        0.035284
num_feature_6         0.025459
num_feature_4         0.007782
num_feature_1         0.005673
num_feature_9         0.005660
num_feature_2         0.004688
cat_region_South      0.001283
cat_segment_B         0.000259
num_feature_8         0.000000
num_feature_5         0.000000
num_feature_15        0.000000
num_feature_12        0.000000
noise_feature_1       0.000000
num_feature_16        0.000000
cat_region_Missing    0.000000
noise_feature_2       0.000000
cat_region_North      0.000000
cat_region_West       0.000000
cat_segment_C         0.000000
cat_flag_Yes          0.000000
dtype: float64


In [87]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

# Identify which columns are actually discrete (one-hot encoded = 0/1 binary)
discrete_mask = [col.startswith("cat_") for col in X_train_scaled.columns]

mi_scores_fixed = mutual_info_classif(
    X_train_scaled, y_train,
    discrete_features=discrete_mask,
    random_state=42
)
mi_series_fixed = pd.Series(mi_scores_fixed, index=X_train_scaled.columns).sort_values(ascending=False)

print("Mutual Information scores (corrected for discrete columns):")
print(mi_series_fixed)

Mutual Information scores (corrected for discrete columns):
num_feature_18        0.177807
num_feature_13        0.082607
num_feature_7         0.080367
num_feature_3         0.050123
num_feature_17        0.046566
num_feature_11        0.044352
num_feature_10        0.036267
num_feature_14        0.035284
num_feature_6         0.025459
num_feature_4         0.007782
num_feature_1         0.005673
num_feature_9         0.005660
num_feature_2         0.004688
cat_flag_Yes          0.000146
cat_region_West       0.000115
cat_region_Missing    0.000078
cat_region_South      0.000005
cat_segment_B         0.000004
cat_segment_C         0.000003
cat_region_North      0.000001
num_feature_5         0.000000
num_feature_8         0.000000
num_feature_15        0.000000
noise_feature_1       0.000000
num_feature_16        0.000000
num_feature_12        0.000000
noise_feature_2       0.000000
dtype: float64


In [88]:
# Select features with MI > 0.001 (natural gap in the data)
mi_selected_features = mi_series_fixed[mi_series_fixed > 0.001].index.tolist()

print("Features selected by MI (threshold > 0.001):", len(mi_selected_features))
print(mi_selected_features)

X_train_mi = X_train_scaled[mi_selected_features]
X_test_mi = X_test_scaled[mi_selected_features]

Features selected by MI (threshold > 0.001): 13
['num_feature_18', 'num_feature_13', 'num_feature_7', 'num_feature_3', 'num_feature_17', 'num_feature_11', 'num_feature_10', 'num_feature_14', 'num_feature_6', 'num_feature_4', 'num_feature_1', 'num_feature_9', 'num_feature_2']


In [89]:
knn_mi = KNeighborsClassifier(n_neighbors=5)
knn_mi.fit(X_train_mi, y_train)

y_pred_mi = knn_mi.predict(X_test_mi)

print("=== KNN (k=5) — MI-Selected (13 features) — Test Set Performance ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_mi), 4))
print("Precision:", round(precision_score(y_test, y_pred_mi), 4))
print("Recall:", round(recall_score(y_test, y_pred_mi), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_mi), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_mi))

print("\n=== Full comparison across all approaches ===")
print(f"{'Metric':<12}{'27 feat':<10}{'17 feat(corr)':<15}{'13 feat(MI)':<12}")
print(f"{'Accuracy':<12}{0.9255:<10}{0.9175:<15}{round(accuracy_score(y_test, y_pred_mi),4):<12}")
print(f"{'Precision':<12}{0.9253:<10}{0.9169:<15}{round(precision_score(y_test, y_pred_mi),4):<12}")
print(f"{'Recall':<12}{0.8586:<10}{0.8430:<15}{round(recall_score(y_test, y_pred_mi),4):<12}")
print(f"{'F1 Score':<12}{0.8907:<10}{0.8784:<15}{round(f1_score(y_test, y_pred_mi),4):<12}")

=== KNN (k=5) — MI-Selected (13 features) — Test Set Performance ===
Accuracy: 0.9375
Precision: 0.9343
Recall: 0.8854
F1 Score: 0.9092

Confusion Matrix:
[[1249   44]
 [  81  626]]

=== Full comparison across all approaches ===
Metric      27 feat   17 feat(corr)  13 feat(MI) 
Accuracy    0.9255    0.9175         0.9375      
Precision   0.9253    0.9169         0.9343      
Recall      0.8586    0.843          0.8854      
F1 Score    0.8907    0.8784         0.9092      
